<!-- # Wood-Powder Segmentation (PointTransformer V3) + Volume Estimation — Full Pipeline Notebook

**Dataset:** 20,000 point clouds (≈400k–800k points each) with binary labels
`0 = environment → RED`, `1 = wood powder → GREEN`, plus a ground-truth volume CSV.

**Notebook sections (run top-to-bottom):**

| # | Section | Purpose |
|---|---------|---------|
| 1 | Setup / installs | dependencies + official PTv3 repo |
| 2 | Config | all paths & hyper-parameters |
| 3 | Utilities | IO (npy/txt/ply/las/pcd), voxel subsampling, metrics, colored PLY export |
| 4 | Split | Train / Val / Test (70/15/15) — **Task 1** |
| 5 | Dataset & Model | PTv3 dataloader + segmentation model |
| 6 | Training | weighted CE + Dice, AMP, OneCycle, best ckpt by val mIoU — **Task 1** |
| 7 | Test + Visualization | TEST-phase only visualization (green/red) + metrics — **Task 1** |
| 8 | Volume features | RANSAC ground plane, height-map integration, voxel occupancy, hull |
| 9 | Feature extraction | run best model over all clouds → `volume_features.csv` |
| 10 | Volume regression | CV model selection (Linear…XGBoost), test error analysis vs GT |
| 11 | **Standalone inference** | load best model → segment 1..N clouds → visualize → volume — **Task 2** |

⚠️ **Units:** `grid_size`, `cell_sizes`, `voxel_sizes_volume`, `ransac_thresh` in the config assume **meters**. If your data is in millimeters, multiply them by 1000.

⚠️ If your file format / label column differs, adapt only `load_point_cloud()` in the Utilities cell. -->

In [41]:
# =====================================================================
# 1. SETUP — run once, then restart the kernel if new packages installed
# =====================================================================
# Uncomment as needed:

# !pip install numpy scipy pandas scikit-learn joblib matplotlib tqdm plyfile open3d laspy addict timm xgboost

# spconv: pick the wheel matching your CUDA version, e.g. CUDA 12.x:
# !pip install spconv-cu120

# torch-scatter: pick the wheel matching your torch + CUDA, e.g.:
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-2.1.0+cu121.html

# flash-attn (OPTIONAL, only if you set CFG["enable_flash"] = True):
# !pip install flash-attn --no-build-isolation

# Official PointTransformer V3 implementation:
# !git clone https://github.com/Pointcept/PointTransformerV3.git third_party/PointTransformerV3

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

torch: 2.8.0+cu126 | CUDA available: True


In [42]:
import pandas as pd

In [43]:
df=pd.read_csv("data/combine_gt.csv")

In [44]:
df.columns

Index(['original_filename', 'modified_filename', 'points', 'volume'], dtype='object')

In [45]:
gt_volume_df=df.drop(columns=["original_filename"])

In [46]:
gt_volume_df.rename(columns={"modified_filename":"filename"},inplace=True)

In [47]:
gt_volume_df.to_csv("gt_volume.csv")

In [48]:
# =====================================================================
# 2. CONFIG — adjust paths & unit-dependent values here
# =====================================================================
CFG = {
    # ---------------- Paths ----------------
    "data_root": "data/train",          # folder with the 20,000 point-cloud files
    "gt_volume_csv": "data/gt_volume.csv",    # ground-truth volume file
    "gt_file_col": "filename",                # column with the file name / id
    "gt_vol_col": "volume",                   # column with the GT volume value
    "ptv3_repo": "data/third_party/PointTransformerV3",  # cloned official PTv3 repo
    "out_dir": "data/output",

    # ---------------- Data / split ----------------
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "seed": 42,

    # ---------------- Segmentation model ----------------
    "num_classes": 2,          # 0 = environment, 1 = target
    "feat_dim": 4,             # x, y, z, height
    "grid_size": 0.02,         # voxel size for grid subsampling fed to PTv3
    "max_points": 120000,      # cap per training sample (after voxelization)
    "max_points_val": 200000,  # cap per validation sample
    "enable_flash": False,     # True only if flash-attn is installed
    "drop_path": 0.3,

    # ---------------- Training ----------------
    "batch_size": 4,
    "epochs": 10,
    "lr": 2e-3,
    "weight_decay": 5e-3,
    "class_weights": [1.0, 2.0],       # [environment, target]
    "train_samples_per_epoch": 4000,   # random subset per epoch (None = all files each epoch)
    "num_workers": 8,
    "amp": True,
    "grad_clip": 10.0,

    # ---------------- Full-cloud inference ----------------
    # 0 = whole voxelized cloud in one pass (GPU >= 12 GB recommended)
    # >0 = slab chunking with at most this many points per forward (small-GPU fallback)
    "max_points_infer": 0,

    # ---------------- Volume features ----------------
    "voxel_sizes_volume": [0.01, 0.02, 0.04],
    "cell_sizes": [0.01, 0.02],
    "ransac_iters": 300,
    "ransac_thresh": 0.01,

    # ---------------- Volume regression ----------------
    "reg_cv_folds": 5,
    "reg_log_target": True,
}
print("Config loaded. out_dir =", CFG["out_dir"])

Config loaded. out_dir = data/output


In [49]:
# =====================================================================
# 3. UTILITIES — file IO, voxel subsampling, metrics, colored PLY export
# =====================================================================
import os
import json
import random

import numpy as np

SUPPORTED_EXTS = (".npy", ".npz", ".txt", ".xyz", ".pts", ".csv", ".ply", ".las", ".laz", ".pcd")

# Segmentation color map (RGB, 0-255)
COLOR_MAP = {
    0: (255, 0, 0),    # environment  -> RED
    1: (0, 255, 0),    # target  -> GREEN
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def file_id(path):
    """Canonical id of a cloud file = basename without extension."""
    return os.path.splitext(os.path.basename(path))[0]


def list_cloud_files(root):
    files = []
    for dirpath, _, names in os.walk(root):
        for n in sorted(names):
            if n.lower().endswith(SUPPORTED_EXTS):
                files.append(os.path.join(dirpath, n))
    return sorted(files)


def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def load_json(path):
    with open(path) as f:
        return json.load(f)


# ------------------------------------------------------------------ #
#  Point-cloud loading — adapt HERE if your data layout differs
# ------------------------------------------------------------------ #
_LABEL_KEYS = ("label", "labels", "class", "classification", "scalar_label",
               "scalar_classification", "seg", "gt")


def load_point_cloud(path, need_labels=True):
    """
    Returns:
        coords : (N, 3) float32 -- ORIGINAL metric coordinates (never normalized here)
        labels : (N,)   int64 or None
    """
    ext = os.path.splitext(path)[1].lower()

    if ext in (".npy", ".npz"):
        arr = np.load(path)
        if isinstance(arr, np.lib.npyio.NpzFile):
            pk = next(k for k in arr.files if k.lower() in ("points", "coords", "xyz", "coord"))
            coords = np.asarray(arr[pk], dtype=np.float32)[:, :3]
            labels = None
            for k in arr.files:
                if k.lower() in _LABEL_KEYS:
                    labels = np.asarray(arr[k]).reshape(-1).astype(np.int64)
                    break
        else:
            arr = np.asarray(arr)
            coords = arr[:, :3].astype(np.float32)
            labels = arr[:, -1].astype(np.int64) if arr.shape[1] > 3 else None

    elif ext in (".txt", ".xyz", ".pts", ".csv"):
        delim = "," if ext == ".csv" else None
        arr = np.loadtxt(path, delimiter=delim, ndmin=2)
        coords = arr[:, :3].astype(np.float32)
        labels = arr[:, -1].astype(np.int64) if arr.shape[1] > 3 else None

    elif ext == ".ply":
        # print("Error")
        from plyfile import PlyData
        ply = PlyData.read(path)
        v = ply["vertex"]
        coords = np.stack([np.asarray(v["x"]), np.asarray(v["y"]), np.asarray(v["z"])],
                          axis=1).astype(np.float32)
        labels = None
        for key in [p.name for p in v.properties]:
            if key.lower() in _LABEL_KEYS:
                labels = np.asarray(v[key]).reshape(-1).astype(np.int64)
                break

    elif ext in (".las", ".laz"):
        import laspy
        las = laspy.read(path)
        coords = np.stack([las.x, las.y, las.z], axis=1).astype(np.float32)
        labels = np.asarray(las.classification).astype(np.int64)

    elif ext == ".pcd":
        import open3d as o3d
        pcd = o3d.io.read_point_cloud(path)
        coords = np.asarray(pcd.points, dtype=np.float32)
        labels = None
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if need_labels and labels is None:
        raise ValueError(f"No label field found in {path}. "
                         f"Adapt load_point_cloud() to your data layout.")
    if labels is not None:
        labels = np.clip(labels, 0, 1)  # binary safety
    return coords, labels


# ------------------------------------------------------------------ #
#  Voxel (grid) subsampling
# ------------------------------------------------------------------ #
def voxel_subsample(coords, grid_size, mode="train"):
    """
    One point per voxel.
      mode='train' : random representative per voxel      -> (idx, None)
      mode='test'  : deterministic representative + full  -> (idx, inverse)
                     'inverse' maps EVERY original point to its voxel index so
                     predictions can be propagated back to all points.
    """
    disc = np.floor(coords / grid_size).astype(np.int64)
    disc -= disc.min(0)
    dims = disc.max(0) + 1
    key = (disc[:, 0] * dims[1] + disc[:, 1]) * dims[2] + disc[:, 2]

    if mode == "train":
        perm = np.random.permutation(key.shape[0])
        _, first = np.unique(key[perm], return_index=True)
        return perm[first], None

    _, first, inverse = np.unique(key, return_index=True, return_inverse=True)
    return first, inverse


# ------------------------------------------------------------------ #
#  Segmentation metrics
# ------------------------------------------------------------------ #
def update_confmat(conf, pred, gt, num_classes):
    mask = (gt >= 0) & (gt < num_classes)
    conf += np.bincount(num_classes * gt[mask] + pred[mask],
                        minlength=num_classes ** 2).reshape(num_classes, num_classes)
    return conf


def metrics_from_confmat(conf):
    tp = np.diag(conf).astype(np.float64)
    fp = conf.sum(0) - tp
    fn = conf.sum(1) - tp
    iou = tp / np.maximum(tp + fp + fn, 1)
    acc = tp.sum() / max(conf.sum(), 1)
    prec = tp / np.maximum(tp + fp, 1)
    rec = tp / np.maximum(tp + fn, 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return {
        "iou_env": iou[0], "iou_powder": iou[1], "miou": iou.mean(),
        "acc": acc,
        "precision_powder": prec[1], "recall_powder": rec[1], "f1_powder": f1[1],
    }


# ------------------------------------------------------------------ #
#  Visualization: colored PLY export + interactive Open3D viewer
#  (RED = environment, GREEN = wood powder)
# ------------------------------------------------------------------ #
def save_colored_ply(path, coords, pred):
    from plyfile import PlyData, PlyElement
    os.makedirs(os.path.dirname(path), exist_ok=True)
    colors = np.zeros((coords.shape[0], 3), dtype=np.uint8)
    for lbl, rgb in COLOR_MAP.items():
        colors[pred == lbl] = rgb
    vertex = np.empty(coords.shape[0], dtype=[("x", "f4"), ("y", "f4"), ("z", "f4"),
                                              ("red", "u1"), ("green", "u1"), ("blue", "u1"),
                                              ("pred", "u1")])
    vertex["x"], vertex["y"], vertex["z"] = coords[:, 0], coords[:, 1], coords[:, 2]
    vertex["red"], vertex["green"], vertex["blue"] = colors[:, 0], colors[:, 1], colors[:, 2]
    vertex["pred"] = pred.astype(np.uint8)
    PlyData([PlyElement.describe(vertex, "vertex")], text=False).write(path)


def show_segmentation_open3d(coords, pred, window_name="segmentation"):
    """Interactive viewer (requires a display)."""
    import open3d as o3d
    colors = np.zeros((coords.shape[0], 3), dtype=np.float64)
    for lbl, rgb in COLOR_MAP.items():
        colors[pred == lbl] = np.array(rgb) / 255.0
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(coords.astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd], window_name=window_name)

print("Utilities ready.")

Utilities ready.


In [50]:
# =====================================================================
# 4. TRAIN / VAL / TEST SPLIT  (Task 1 — part 1)
# =====================================================================
def make_or_load_split(cfg=CFG):
    split_path = os.path.join(cfg["out_dir"], "split.json")
    if os.path.exists(split_path):
        return load_json(split_path)

    set_seed(cfg["seed"])
    files = list_cloud_files(cfg["data_root"])
    if not files:
        raise RuntimeError(f"No point-cloud files found under {cfg['data_root']}")

    files = np.array(files)
    perm = np.random.permutation(len(files))
    n_test = int(round(len(files) * cfg["test_ratio"]))
    n_val = int(round(len(files) * cfg["val_ratio"]))

    split = {
        "test": files[perm[:n_test]].tolist(),
        "val": files[perm[n_test:n_test + n_val]].tolist(),
        "train": files[perm[n_test + n_val:]].tolist(),
    }
    save_json(split, split_path)
    print(f"Split saved to {split_path}: "
          f"train={len(split['train'])}, val={len(split['val'])}, test={len(split['test'])}")
    return split


split = make_or_load_split(CFG)
print({k: len(v) for k, v in split.items()})

Split saved to data/output/split.json: train=31, val=7, test=7
{'test': 7, 'val': 7, 'train': 31}


In [51]:
# =====================================================================
# 5a. DATASET + COLLATE (PointTransformer V3 input format)
# =====================================================================
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler


def _augment(coords):
    theta = np.random.uniform(0.0, 2.0 * np.pi)
    c, s = np.cos(theta), np.sin(theta)
    rot = np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    coords = coords @ rot.T
    coords *= np.random.uniform(0.9, 1.1)
    if np.random.rand() < 0.5:
        coords[:, 0] = -coords[:, 0]
    if np.random.rand() < 0.5:
        coords[:, 1] = -coords[:, 1]
    coords += np.random.normal(0.0, 0.003, size=coords.shape).astype(np.float32)
    return coords


class WoodPowderDataset(Dataset):
    """Each item = one full point cloud, normalized + voxel-subsampled."""

    def __init__(self, files, cfg, split="train"):
        self.files = list(files)
        self.cfg = cfg
        self.split = split

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        cfg = self.cfg
        coords, labels = load_point_cloud(self.files[idx], need_labels=True)
        coords = coords.astype(np.float32)

        # center xy at mean, floor z at 0 (training never needs original coords)
        coords[:, 0] -= coords[:, 0].mean()
        coords[:, 1] -= coords[:, 1].mean()
        coords[:, 2] -= coords[:, 2].min()

        if self.split == "train":
            coords = _augment(coords)

        mode = "train" if self.split == "train" else "test"
        sel, _ = voxel_subsample(coords, cfg["grid_size"], mode=mode)
        coords, labels = coords[sel], labels[sel]

        cap = cfg["max_points"] if self.split == "train" else cfg["max_points_val"]
        if cap and coords.shape[0] > cap:
            keep = np.random.choice(coords.shape[0], cap, replace=False)
            coords, labels = coords[keep], labels[keep]

        coords -= coords.min(0)  # non-negative for grid coords
        grid_coord = np.floor(coords / cfg["grid_size"]).astype(np.int64)
        feat = np.concatenate([coords, coords[:, 2:3]], axis=1).astype(np.float32)  # xyz + height

        return {
            "coord": torch.from_numpy(coords),
            "grid_coord": torch.from_numpy(grid_coord),
            "feat": torch.from_numpy(feat),
            "label": torch.from_numpy(labels.astype(np.int64)),
        }


def collate_fn(batch):
    counts = torch.tensor([b["coord"].shape[0] for b in batch], dtype=torch.long)
    return {
        "coord": torch.cat([b["coord"] for b in batch], dim=0),
        "grid_coord": torch.cat([b["grid_coord"] for b in batch], dim=0),
        "feat": torch.cat([b["feat"] for b in batch], dim=0),
        "label": torch.cat([b["label"] for b in batch], dim=0),
        "offset": torch.cumsum(counts, dim=0).long(),
    }

print("Dataset ready.")

Dataset ready.


In [52]:
# =====================================================================
# 5b. POINTTRANSFORMER V3 SEGMENTATION MODEL
#     (uses the OFFICIAL implementation — see setup cell for the git clone)
# =====================================================================
import sys
import torch.nn as nn


def _import_ptv3(repo_path):
    if repo_path and repo_path not in sys.path:
        sys.path.insert(0, repo_path)
    try:
        # standalone repo: <repo>/model.py
        from model import PointTransformerV3  # noqa
        return PointTransformerV3
    except ImportError:
        # full Pointcept repo fallback
        from pointcept.models.point_transformer_v3.point_transformer_v3m1_base import (  # noqa
            PointTransformerV3,
        )
        return PointTransformerV3


class PTv3Segmentor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        ptv3_cls = _import_ptv3(cfg.get("ptv3_repo"))
        self.backbone = ptv3_cls(
            in_channels=cfg["feat_dim"],
            cls_mode=False,                          # U-Net mode -> per-point features
            enable_flash=cfg.get("enable_flash", False),
            drop_path=cfg.get("drop_path", 0.3),
        )
        # default PTv3 decoder output dim = 64
        self.head = nn.Sequential(
            nn.Linear(64, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, cfg["num_classes"]),
        )

    def forward(self, data_dict):
        point = self.backbone(data_dict)
        feat = point.feat if hasattr(point, "feat") else point["feat"]
        return self.head(feat)  # (N, num_classes) logits


def build_model(cfg):
    return PTv3Segmentor(cfg)

print("Model definition ready.")

Model definition ready.


In [53]:
# =====================================================================
# 5c. CORE FULL-CLOUD INFERENCE
#     voxel-subsample -> PTv3 forward -> map predictions back to ALL points
# =====================================================================
def load_checkpoint(ckpt_path, device="cuda"):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    cfg = ckpt["cfg"]
    model = build_model(cfg).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model, cfg


def _amp_ctx_infer(device, enabled):
    if not enabled or not str(device).startswith("cuda"):
        return torch.autocast(device_type="cpu", enabled=False)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast(device_type="cuda", dtype=dtype, enabled=True)


@torch.no_grad()
def _forward_points(model, cfg, coords, device):
    """Forward one contiguous set of (already voxelized) points -> per-point labels."""
    coords = coords - coords.min(0)
    grid_coord = np.floor(coords / cfg["grid_size"]).astype(np.int64)
    feat = np.concatenate([coords, coords[:, 2:3]], axis=1).astype(np.float32)

    data = {
        "coord": torch.from_numpy(coords).to(device),
        "grid_coord": torch.from_numpy(grid_coord).to(device),
        "feat": torch.from_numpy(feat).to(device),
        "offset": torch.tensor([coords.shape[0]], dtype=torch.long, device=device),
    }
    with _amp_ctx_infer(device, cfg.get("amp", True)):
        logits = model(data)
    return logits.float().argmax(dim=1).cpu().numpy().astype(np.int64)


@torch.no_grad()
def segment_full_cloud(model, cfg, coords_orig, device="cuda"):
    """
    coords_orig : (N, 3) ORIGINAL metric coordinates (400k-800k points).
    Returns pred : (N,) int64 label per ORIGINAL point (0=env/red, 1=powder/green).
    """
    coords = coords_orig.astype(np.float32).copy()
    coords[:, 0] -= coords[:, 0].mean()
    coords[:, 1] -= coords[:, 1].mean()
    coords[:, 2] -= coords[:, 2].min()

    sel, inverse = voxel_subsample(coords, cfg["grid_size"], mode="test")
    c = coords[sel]

    max_pts = int(cfg.get("max_points_infer", 0) or 0)
    if max_pts > 0 and c.shape[0] > max_pts:
        # slab chunking along the longest horizontal axis (small-GPU fallback)
        axis = int(np.argmax(c[:, :2].max(0) - c[:, :2].min(0)))
        order = np.argsort(c[:, axis], kind="stable")
        pred_vox = np.empty(c.shape[0], dtype=np.int64)
        n_chunks = int(np.ceil(c.shape[0] / max_pts))
        size = int(np.ceil(c.shape[0] / n_chunks))
        for k in range(n_chunks):
            part = order[k * size:(k + 1) * size]
            pred_vox[part] = _forward_points(model, cfg, c[part], device)
    else:
        pred_vox = _forward_points(model, cfg, c, device)

    return pred_vox[inverse]  # propagate to every original point

print("Inference core ready.")

Inference core ready.


In [54]:
# =====================================================================
# 6. TRAINING  (Task 1 — part 2)
#    weighted CrossEntropy + Dice, AMP, OneCycle LR,
#    best checkpoint by validation mIoU -> outputs/checkpoints/best_model.pth
# =====================================================================
import csv
import time
import torch.nn.functional as F


class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, logits, target):
        num_classes = logits.shape[1]
        probs = F.softmax(logits, dim=1)
        onehot = F.one_hot(target, num_classes).float()
        inter = (probs * onehot).sum(0)
        denom = probs.sum(0) + onehot.sum(0)
        dice = (2 * inter + self.eps) / (denom + self.eps)
        return 1.0 - dice.mean()


def _amp_ctx_train(enabled):
    if not enabled:
        return torch.autocast(device_type="cuda", enabled=False)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast(device_type="cuda", dtype=dtype, enabled=True)


def _to_device(batch, device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def train_one_epoch(model, loader, opt, sched, scaler, ce, dice, cfg, device, epoch):
    model.train()
    total, n_iter = 0.0, 0
    t0 = time.time()
    for it, batch in enumerate(loader):
        batch = _to_device(batch, device)
        label = batch.pop("label")

        opt.zero_grad(set_to_none=True)
        with _amp_ctx_train(cfg["amp"]):
            logits = model(batch)
            loss = ce(logits, label) + dice(logits, label)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
        sched.step()

        total += loss.item()
        n_iter += 1
        if it % 50 == 0:
            print(f"[epoch {epoch}] iter {it}/{len(loader)} "
                  f"loss={loss.item():.4f} lr={sched.get_last_lr()[0]:.2e} "
                  f"({time.time() - t0:.0f}s)")
    return total / max(n_iter, 1)


@torch.no_grad()
def validate(model, loader, cfg, device):
    model.eval()
    conf = np.zeros((cfg["num_classes"], cfg["num_classes"]), dtype=np.int64)
    for batch in loader:
        batch = _to_device(batch, device)
        label = batch.pop("label")
        with _amp_ctx_train(cfg["amp"]):
            logits = model(batch)
        pred = logits.float().argmax(1).cpu().numpy()
        conf = update_confmat(conf, pred, label.cpu().numpy(), cfg["num_classes"])
    return metrics_from_confmat(conf)


def run_training(cfg=CFG):
    set_seed(cfg["seed"])
    device = "cuda"
    assert torch.cuda.is_available(), "CUDA GPU is required for PTv3 training."

    split = make_or_load_split(cfg)
    train_ds = WoodPowderDataset(split["train"], cfg, split="train")
    val_ds = WoodPowderDataset(split["val"], cfg, split="val")

    sampler = None
    if cfg.get("train_samples_per_epoch"):
        sampler = RandomSampler(train_ds, replacement=True,
                                num_samples=cfg["train_samples_per_epoch"])

    train_loader = DataLoader(
        train_ds, batch_size=cfg["batch_size"],
        shuffle=(sampler is None), sampler=sampler,
        num_workers=cfg["num_workers"], collate_fn=collate_fn,
        pin_memory=True, drop_last=True, persistent_workers=cfg["num_workers"] > 0)
    val_loader = DataLoader(
        val_ds, batch_size=1, shuffle=False,
        num_workers=max(cfg["num_workers"] // 2, 1), collate_fn=collate_fn,
        pin_memory=True)

    model = build_model(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"PTv3 segmentor built: {n_params:.1f}M params, "
          f"train={len(train_ds)} val={len(val_ds)} clouds")

    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                            weight_decay=cfg["weight_decay"])
    total_steps = cfg["epochs"] * len(train_loader)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg["lr"], total_steps=total_steps, pct_start=0.05,
        anneal_strategy="cos", div_factor=10.0, final_div_factor=100.0)

    use_fp16_scaler = cfg["amp"] and not torch.cuda.is_bf16_supported()
    scaler = torch.cuda.amp.GradScaler() if use_fp16_scaler else None

    ce = nn.CrossEntropyLoss(
        weight=torch.tensor(cfg["class_weights"], dtype=torch.float32, device=device))
    dice = DiceLoss()

    ckpt_dir = os.path.join(cfg["out_dir"], "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    log_path = os.path.join(cfg["out_dir"], "train_log.csv")
    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "val_miou", "val_iou_powder",
                                "val_iou_env", "val_acc", "val_f1_powder"])

    best_miou = -1.0
    for epoch in range(1, cfg["epochs"] + 1):
        loss = train_one_epoch(model, train_loader, opt, sched, scaler,
                               ce, dice, cfg, device, epoch)
        m = validate(model, val_loader, cfg, device)
        print(f"== epoch {epoch}: loss={loss:.4f}  mIoU={m['miou']:.4f}  "
              f"IoU(powder)={m['iou_powder']:.4f}  IoU(env)={m['iou_env']:.4f}  "
              f"acc={m['acc']:.4f}")

        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow([epoch, f"{loss:.5f}", f"{m['miou']:.5f}",
                                    f"{m['iou_powder']:.5f}", f"{m['iou_env']:.5f}",
                                    f"{m['acc']:.5f}", f"{m['f1_powder']:.5f}"])

        state = {"model": model.state_dict(), "cfg": cfg,
                 "epoch": epoch, "val_metrics": m}
        torch.save(state, os.path.join(ckpt_dir, "last_model.pth"))
        if m["miou"] > best_miou:
            best_miou = m["miou"]
            torch.save(state, os.path.join(ckpt_dir, "best_model.pth"))
            print(f"   -> new BEST model saved (mIoU={best_miou:.4f})")

    print(f"Training done. Best val mIoU = {best_miou:.4f}")

print("Training utilities ready.")

Training utilities ready.


In [55]:
# --- RUN TRAINING (long-running) ---
run_training(CFG)

ModuleNotFoundError: No module named 'pointcept'

In [ ]:
# =====================================================================
# 7. TEST-PHASE EVALUATION + VISUALIZATION  (Task 1 — part 3)
#    Visualization happens ONLY here (test phase), as required:
#       wood powder (1) -> GREEN,  everything else (0) -> RED
# =====================================================================
def test_evaluation(ckpt=None, show=0, limit=0, device="cuda", cfg=CFG):
    """
    ckpt  : checkpoint path (default: outputs/checkpoints/best_model.pth)
    show  : open interactive Open3D windows for the first N test clouds
    limit : evaluate only the first N test files (0 = all)
    """
    ckpt = ckpt or os.path.join(cfg["out_dir"], "checkpoints", "best_model.pth")
    model, mcfg = load_checkpoint(ckpt, device)
    split = make_or_load_split(cfg)
    test_files = split["test"][:limit] if limit else split["test"]

    vis_dir = os.path.join(cfg["out_dir"], "test_vis")
    os.makedirs(vis_dir, exist_ok=True)

    conf_total = np.zeros((mcfg["num_classes"], mcfg["num_classes"]), dtype=np.int64)
    per_file_path = os.path.join(cfg["out_dir"], "test_metrics_per_file.csv")
    with open(per_file_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["file_id", "n_points", "miou", "iou_powder", "iou_env",
                         "acc", "f1_powder"])

        for i, path in enumerate(test_files):
            coords, labels = load_point_cloud(path, need_labels=True)
            pred = segment_full_cloud(model, mcfg, coords, device)

            conf = np.zeros_like(conf_total)
            conf = update_confmat(conf, pred, labels, mcfg["num_classes"])
            conf_total += conf
            m = metrics_from_confmat(conf)

            fid = file_id(path)
            save_colored_ply(os.path.join(vis_dir, f"{fid}_pred.ply"), coords, pred)
            writer.writerow([fid, len(coords), f"{m['miou']:.5f}",
                             f"{m['iou_powder']:.5f}", f"{m['iou_env']:.5f}",
                             f"{m['acc']:.5f}", f"{m['f1_powder']:.5f}"])
            print(f"[{i + 1}/{len(test_files)}] {fid}: mIoU={m['miou']:.4f} "
                  f"IoU(powder)={m['iou_powder']:.4f}")

            if i < show:
                show_segmentation_open3d(coords, pred, window_name=fid)

    m = metrics_from_confmat(conf_total)
    print("\n================ TEST SET (overall) ================")
    for k, v in m.items():
        print(f"{k:>18s}: {v:.4f}")
    print(f"Colored predictions saved in: {vis_dir}")
    print(f"Per-file metrics: {per_file_path}")
    return m

print("Test-evaluation utilities ready.")

In [ ]:
# --- RUN TEST EVALUATION + VISUALIZATION ---
# show=3 opens 3 interactive 3D windows (needs a display); on a headless server keep show=0
# and open the saved .ply files locally (CloudCompare / MeshLab / Open3D).
test_metrics = test_evaluation(show=0, limit=0)

In [ ]:
# =====================================================================
# 8. GEOMETRIC VOLUME FEATURES (from segmented wood-powder points,
#    computed in ORIGINAL metric coordinates)
#    1) 2.5D height-map integration above a RANSAC ground plane
#    2) multi-resolution voxel-occupancy volume
#    3) convex-hull volume / area
#    4) shape statistics
# =====================================================================
from collections import OrderedDict


def fit_plane_ransac(pts, iters=300, thresh=0.01, max_pts=30000, seed=0):
    """Returns (n, d) with unit normal n so that the plane is n·x + d = 0."""
    rng = np.random.default_rng(seed)
    if pts.shape[0] < 50:
        return np.array([0.0, 0.0, 1.0]), (-float(pts[:, 2].min()) if len(pts) else 0.0)

    if pts.shape[0] > max_pts:
        pts = pts[rng.choice(pts.shape[0], max_pts, replace=False)]

    best_n, best_d, best_cnt = np.array([0.0, 0.0, 1.0]), -float(pts[:, 2].min()), -1
    for _ in range(iters):
        p = pts[rng.choice(pts.shape[0], 3, replace=False)]
        n = np.cross(p[1] - p[0], p[2] - p[0])
        norm = np.linalg.norm(n)
        if norm < 1e-9:
            continue
        n /= norm
        d = -float(n @ p[0])
        cnt = int((np.abs(pts @ n + d) < thresh).sum())
        if cnt > best_cnt:
            best_cnt, best_n, best_d = cnt, n, d

    # least-squares refinement on inliers
    inl = pts[np.abs(pts @ best_n + best_d) < thresh]
    if inl.shape[0] >= 3:
        centroid = inl.mean(0)
        _, _, vh = np.linalg.svd(inl - centroid, full_matrices=False)
        n = vh[-1]
        n /= np.linalg.norm(n)
        best_n, best_d = n, -float(n @ centroid)
    return best_n, best_d


def heights_above_plane(pts, n, d, sign_ref=None):
    h = pts @ n + d
    ref = sign_ref if sign_ref is not None else pts
    if np.median(ref @ n + d) < 0:  # orient normal so the pile is on the + side
        h = -h
    return np.clip(h, 0.0, None)


def heightmap_volume(xy, h, cell):
    """Returns (vol_max_surface, vol_mean_surface, footprint_area)."""
    if xy.shape[0] == 0:
        return 0.0, 0.0, 0.0
    ij = np.floor(xy / cell).astype(np.int64)
    ij -= ij.min(0)
    key = ij[:, 0] * (ij[:, 1].max() + 1) + ij[:, 1]
    uniq, inv = np.unique(key, return_inverse=True)

    h_max = np.zeros(uniq.shape[0], dtype=np.float64)
    np.maximum.at(h_max, inv, h)
    h_sum = np.zeros(uniq.shape[0], dtype=np.float64)
    np.add.at(h_sum, inv, h)
    cnt = np.bincount(inv, minlength=uniq.shape[0]).astype(np.float64)
    h_mean = h_sum / np.maximum(cnt, 1)

    area = cell * cell
    return float(h_max.sum() * area), float(h_mean.sum() * area), float(uniq.shape[0] * area)


def feature_names(cfg):
    names = ["n_powder_points", "h_max", "h_mean", "h_p95", "bbox_volume"]
    for c in cfg["cell_sizes"]:
        names += [f"hm_vol_max_{c}", f"hm_vol_mean_{c}", f"footprint_{c}"]
    for v in cfg["voxel_sizes_volume"]:
        names += [f"voxel_vol_{v}"]
    names += ["hull_volume", "hull_area"]
    return names


def compute_volume_features(powder_pts, env_pts, cfg):
    f = OrderedDict((k, 0.0) for k in feature_names(cfg))
    f["n_powder_points"] = float(powder_pts.shape[0])
    if powder_pts.shape[0] < 10:
        return f

    # ground plane from environment points (fallback: powder min-z plane)
    if env_pts is not None and env_pts.shape[0] >= 50:
        n, d = fit_plane_ransac(env_pts, cfg["ransac_iters"], cfg["ransac_thresh"],
                                seed=cfg["seed"])
    else:
        n, d = np.array([0.0, 0.0, 1.0]), -float(powder_pts[:, 2].min())
    h = heights_above_plane(powder_pts, n, d, sign_ref=powder_pts)

    f["h_max"] = float(h.max())
    f["h_mean"] = float(h.mean())
    f["h_p95"] = float(np.percentile(h, 95))

    ext = powder_pts.max(0) - powder_pts.min(0)
    f["bbox_volume"] = float(ext[0] * ext[1] * max(h.max(), 1e-9))

    # 2.5D height-map volumes in the ground-plane basis (u, v)
    a = np.array([1.0, 0.0, 0.0])
    if abs(n @ a) > 0.9:
        a = np.array([0.0, 1.0, 0.0])
    u = np.cross(n, a); u /= np.linalg.norm(u)
    v = np.cross(n, u)
    xy = np.stack([powder_pts @ u, powder_pts @ v], axis=1)

    for c in cfg["cell_sizes"]:
        vmax, vmean, area = heightmap_volume(xy, h, c)
        f[f"hm_vol_max_{c}"] = vmax
        f[f"hm_vol_mean_{c}"] = vmean
        f[f"footprint_{c}"] = area

    # voxel occupancy volumes
    for vsize in cfg["voxel_sizes_volume"]:
        disc = np.floor(powder_pts / vsize).astype(np.int64)
        disc -= disc.min(0)
        key = (disc[:, 0] * (disc[:, 1].max() + 1) + disc[:, 1]) * (disc[:, 2].max() + 1) + disc[:, 2]
        f[f"voxel_vol_{vsize}"] = float(np.unique(key).shape[0]) * vsize ** 3

    # convex hull
    try:
        from scipy.spatial import ConvexHull
        sub = powder_pts
        if sub.shape[0] > 100000:
            sub = sub[np.random.default_rng(cfg["seed"]).choice(sub.shape[0], 100000, replace=False)]
        hull = ConvexHull(sub)
        f["hull_volume"] = float(hull.volume)
        f["hull_area"] = float(hull.area)
    except Exception:
        pass  # degenerate cloud -> keep zeros

    return f


def physics_volume_estimate(features, cfg):
    """Regression-free fallback: finest max-surface height-map integration."""
    finest = min(cfg["cell_sizes"])
    return float(features[f"hm_vol_max_{finest}"])

print("Volume feature utilities ready.")

In [ ]:
# =====================================================================
# 9. FEATURE EXTRACTION — run the best segmentation model over the
#    dataset and store volume features -> outputs/volume_features.csv
# =====================================================================
def extract_features(ckpt=None, splits=("train", "val", "test"),
                     use_gt_labels=False, device="cuda", cfg=CFG):
    """
    use_gt_labels=True computes the features from the GT masks instead of
    the predicted masks (useful as an upper-bound sanity check).
    """
    split = make_or_load_split(cfg)
    if use_gt_labels:
        model, mcfg = None, cfg
    else:
        ckpt = ckpt or os.path.join(cfg["out_dir"], "checkpoints", "best_model.pth")
        model, mcfg = load_checkpoint(ckpt, device)

    out_csv = os.path.join(cfg["out_dir"], "volume_features.csv")
    names = feature_names(cfg)
    os.makedirs(cfg["out_dir"], exist_ok=True)

    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["file_id", "split"] + names)

        for sp in splits:
            files = split[sp]
            for i, path in enumerate(files):
                coords, labels = load_point_cloud(path, need_labels=use_gt_labels)
                if use_gt_labels:
                    mask = labels == 1
                else:
                    pred = segment_full_cloud(model, mcfg, coords, device)
                    mask = pred == 1

                feats = compute_volume_features(coords[mask], coords[~mask], cfg)
                writer.writerow([file_id(path), sp] + [f"{feats[k]:.8g}" for k in names])

                if (i + 1) % 50 == 0 or i + 1 == len(files):
                    print(f"[{sp}] {i + 1}/{len(files)} done")

    print(f"Features saved to {out_csv}")
    return out_csv


# --- RUN FEATURE EXTRACTION (long-running: full dataset inference) ---
extract_features()

In [ ]:
# =====================================================================
# 10. VOLUME REGRESSION — candidate comparison (K-fold CV on train+val),
#     best-model selection, TEST error analysis vs GT volume file
# =====================================================================
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, HuberRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import KFold, cross_val_score


def regression_metrics(y_true, y_pred):
    err = y_pred - y_true
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    denom = np.maximum(np.abs(y_true), 1e-12)
    mape = float(np.mean(np.abs(err) / denom) * 100.0)
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = 1.0 - ss_res / max(ss_tot, 1e-12)
    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape, "R2": r2}


def build_candidates(cfg):
    seed = cfg["seed"]
    base = {
        "Linear": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=seed),
        "Huber": HuberRegressor(max_iter=2000),
        "RandomForest": RandomForestRegressor(
            n_estimators=500, min_samples_leaf=2, n_jobs=-1, random_state=seed),
        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=500, min_samples_leaf=2, n_jobs=-1, random_state=seed),
        "GradientBoosting": GradientBoostingRegressor(
            n_estimators=600, learning_rate=0.05, max_depth=3,
            subsample=0.8, random_state=seed),
    }
    try:
        from xgboost import XGBRegressor
        base["XGBoost"] = XGBRegressor(
            n_estimators=800, learning_rate=0.05, max_depth=5,
            subsample=0.8, colsample_bytree=0.8,
            random_state=seed, n_jobs=-1, tree_method="hist")
    except ImportError:
        print("xgboost not installed -> skipping XGBoost candidate")

    candidates = {}
    for name, est in base.items():
        candidates[name] = Pipeline([("scaler", StandardScaler()), ("reg", est)])
        if cfg["reg_log_target"]:
            candidates[name + "_logY"] = TransformedTargetRegressor(
                regressor=Pipeline([("scaler", StandardScaler()),
                                    ("reg", type(est)(**est.get_params()))]),
                func=np.log1p, inverse_func=np.expm1)
    return candidates


def run_regression(cfg=CFG):
    out_dir = os.path.join(cfg["out_dir"], "regression")
    os.makedirs(out_dir, exist_ok=True)

    # ---- load & merge features with GT volumes ----
    feats = pd.read_csv(os.path.join(cfg["out_dir"], "volume_features.csv"))
    gt = pd.read_csv(cfg["gt_volume_csv"])
    gt = gt.rename(columns={cfg["gt_file_col"]: "file_id", cfg["gt_vol_col"]: "gt_volume"})
    gt["file_id"] = gt["file_id"].astype(str).map(
        lambda s: os.path.splitext(os.path.basename(s))[0])
    feats["file_id"] = feats["file_id"].astype(str)

    df = feats.merge(gt[["file_id", "gt_volume"]], on="file_id", how="inner")
    print(f"Merged samples: {len(df)} (features={len(feats)}, gt={len(gt)})")
    if len(df) == 0:
        raise RuntimeError("No overlap between feature file_ids and GT file names. "
                           "Check gt_file_col / file naming.")

    cols = feature_names(cfg)
    trainval = df[df["split"].isin(["train", "val"])]
    test = df[df["split"] == "test"]
    X_tr, y_tr = trainval[cols].values, trainval["gt_volume"].values.astype(np.float64)
    X_te, y_te = test[cols].values, test["gt_volume"].values.astype(np.float64)

    # ---- model selection by K-fold CV RMSE on train+val ----
    kf = KFold(n_splits=cfg["reg_cv_folds"], shuffle=True, random_state=cfg["seed"])
    rows, best_name, best_rmse, best_model = [], None, np.inf, None
    for name, model in build_candidates(cfg).items():
        scores = cross_val_score(model, X_tr, y_tr, cv=kf,
                                 scoring="neg_root_mean_squared_error", n_jobs=1)
        rmse = float(-scores.mean())
        rows.append({"model": name, "cv_RMSE": rmse, "cv_RMSE_std": float(scores.std())})
        print(f"{name:>24s}: CV RMSE = {rmse:.6g} (+/- {scores.std():.4g})")
        if rmse < best_rmse:
            best_name, best_rmse, best_model = name, rmse, model

    pd.DataFrame(rows).sort_values("cv_RMSE").to_csv(
        os.path.join(out_dir, "model_comparison.csv"), index=False)

    # ---- refit best model on train+val, evaluate on TEST ----
    best_model.fit(X_tr, y_tr)
    y_pred = best_model.predict(X_te)
    m = regression_metrics(y_te, y_pred)

    finest = min(cfg["cell_sizes"])
    y_phys = test[f"hm_vol_max_{finest}"].values.astype(np.float64)
    m_phys = regression_metrics(y_te, y_phys)

    print(f"\nBEST regressor: {best_name} (CV RMSE={best_rmse:.6g})")
    print("TEST metrics (regression):", {k: round(v, 6) for k, v in m.items()})
    print("TEST metrics (physics height-map baseline):",
          {k: round(v, 6) for k, v in m_phys.items()})

    # ---- per-sample error analysis ----
    err = pd.DataFrame({
        "file_id": test["file_id"].values,
        "gt_volume": y_te,
        "pred_volume": y_pred,
        "abs_error": np.abs(y_pred - y_te),
        "rel_error_%": np.abs(y_pred - y_te) / np.maximum(np.abs(y_te), 1e-12) * 100.0,
    }).sort_values("abs_error", ascending=False)
    err.to_csv(os.path.join(out_dir, "test_predictions.csv"), index=False)

    # ---- plots (shown inline + saved) ----
    plt.figure(figsize=(6, 6))
    plt.scatter(y_te, y_pred, s=8, alpha=0.5)
    lim = [min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())]
    plt.plot(lim, lim, "k--", lw=1)
    plt.xlabel("GT volume"); plt.ylabel("Predicted volume")
    plt.title(f"{best_name} - R2={m['R2']:.4f}, MAPE={m['MAPE_%']:.2f}%")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, "pred_vs_gt.png"), dpi=150)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.hist(y_pred - y_te, bins=60)
    plt.xlabel("residual (pred - GT)"); plt.ylabel("count")
    plt.title("Residual distribution (test)")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, "residuals.png"), dpi=150)
    plt.show()

    joblib.dump({"model": best_model, "feature_names": cols,
                 "name": best_name, "test_metrics": m},
                os.path.join(out_dir, "best_regressor.joblib"))
    print(f"Saved: {os.path.join(out_dir, 'best_regressor.joblib')}")
    return best_name, m


# --- RUN VOLUME-REGRESSION TRAINING + ERROR ANALYSIS ---
run_regression(CFG)

In [ ]:
# =====================================================================
# 11. STANDALONE INFERENCE PIPELINE  (Task 2)
#     * loads the saved BEST segmentation checkpoint
#     * runs inference on ONE file or EVERY file in a directory
#     * visualizes / exports segmentation (GREEN = powder, RED = others)
#     * estimates the wood-powder VOLUME
#         - trained regressor if best_regressor.joblib exists,
#         - physics height-map estimate otherwise
# =====================================================================
def _load_regressor(path):
    if path and os.path.exists(path):
        bundle = joblib.load(path)
        print(f"Volume regressor loaded: {bundle['name']}")
        return bundle
    print("No regressor found -> using physics height-map volume estimate.")
    return None


def _estimate_volume(coords, pred, cfg, regressor):
    powder, env = coords[pred == 1], coords[pred == 0]
    feats = compute_volume_features(powder, env, cfg)
    if regressor is not None:
        x = np.array([[feats[k] for k in regressor["feature_names"]]], dtype=np.float64)
        vol = float(regressor["model"].predict(x)[0])
        method = f"regression:{regressor['name']}"
    else:
        vol = physics_volume_estimate(feats, cfg)
        method = "physics:heightmap"
    return max(vol, 0.0), method, feats


def run_inference(input_path, ckpt=None, regressor_path=None, out_dir=None,
                  show=False, save_ply=True, device="cuda", cfg=CFG):
    """
    input_path : a single point-cloud file OR a directory of files
    show       : open interactive Open3D window(s) (needs a display)
    Returns a list of dicts (one per file) and writes inference_results.csv
    """
    ckpt = ckpt or os.path.join(cfg["out_dir"], "checkpoints", "best_model.pth")
    regressor_path = regressor_path or os.path.join(
        cfg["out_dir"], "regression", "best_regressor.joblib")
    out_dir = out_dir or os.path.join(cfg["out_dir"], "inference")

    files = list_cloud_files(input_path) if os.path.isdir(input_path) else [input_path]
    if not files:
        raise RuntimeError(f"No point-cloud files found at {input_path}")

    model, mcfg = load_checkpoint(ckpt, device)
    regressor = _load_regressor(regressor_path)
    os.makedirs(out_dir, exist_ok=True)

    results = []
    results_csv = os.path.join(out_dir, "inference_results.csv")
    with open(results_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["file_id", "n_points", "n_powder_points",
                         "powder_ratio_%", "estimated_volume", "method"])

        for i, path in enumerate(files):
            coords, _ = load_point_cloud(path, need_labels=False)

            # 1) segmentation on the FULL-resolution cloud
            pred = segment_full_cloud(model, mcfg, coords, device)

            # 2) volume estimation from the segmented powder points
            vol, method, _ = _estimate_volume(coords, pred, cfg, regressor)

            fid = file_id(path)
            n_powder = int((pred == 1).sum())
            ratio = 100.0 * n_powder / max(len(pred), 1)
            print(f"[{i + 1}/{len(files)}] {fid}: points={len(pred)}, "
                  f"powder={n_powder} ({ratio:.1f}%), "
                  f"estimated volume={vol:.6g}  ({method})")

            writer.writerow([fid, len(pred), n_powder, f"{ratio:.3f}",
                             f"{vol:.8g}", method])
            results.append({"file_id": fid, "n_points": len(pred),
                            "n_powder_points": n_powder,
                            "estimated_volume": vol, "method": method})

            # 3) visualization: green = wood powder, red = others
            if save_ply:
                save_colored_ply(os.path.join(out_dir, f"{fid}_pred.ply"), coords, pred)
            if show:
                show_segmentation_open3d(coords, pred,
                                         window_name=f"{fid} | V={vol:.4g}")

    print(f"\nDone. Results: {results_csv}")
    if save_ply:
        print(f"Colored predictions: {out_dir}/<file_id>_pred.ply")
    return results

print("Standalone inference pipeline ready.")

In [ ]:
# --- RUN STANDALONE INFERENCE (Task 2) ---
# Single file:
# results = run_inference("path/to/cloud.ply", show=True)

# Whole folder (headless server -> show=False, open the saved PLYs locally):
# results = run_inference("path/to/folder", show=False)

# Example: run on the first test-split cloud
results = run_inference(make_or_load_split(CFG)["test"][0], show=False)
results